# Experiments

This notebook repeats the main experiments from our report with the final code in `src/`.
Everything is measured with the same CV (stratified 5-fold, repeated 3 times, see `src/validation.py`),
so all numbers can be compared with each other.

Our older notebooks (v1 and v2 features, the first error analysis) are in our first repository.
Here we start from v3, because the final feature file still contains those features.

Put `train.jsonl` and `type_chart_gen2.csv` in `data/` first.
It takes about 10 minutes.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath("."))
from src.submission_best_features import (build_features, load_healing_moves, load_identity_vocab, load_species,
                          load_type_chart, select_features)
from src.submission_best_features_teammate import build_pokedex, load_setup_moves
from src.validation import make_cv

DATA = "data"  # folder with train.jsonl and type_chart_gen2.csv

## Features

We build the full feature table once (217 columns) and then pick columns from it for each experiment.

In [2]:
train = pd.read_json(os.path.join(DATA, "train.jsonl"), lines=True)
vocab = load_identity_vocab([train])
X = build_features(train, load_species([train]), load_type_chart(os.path.join(DATA, "type_chart_gen2.csv")),
                   load_healing_moves([train]), vocab, build_pokedex(train), load_setup_moves([train]))
y = train.set_index("battle_id").loc[X.index, "player_won"]
print(X.shape, "P1 win rate:", round(y.mean(), 3))

(10000, 217) P1 win rate: 0.5


In [3]:
# three groups of columns
identity_cols = [f"{side}_alive_{name}" for name in vocab for side in ("p1", "p2")]
teammate_cols = [c for c in X.columns if c.startswith("f_")]
our_cols = [c for c in X.columns if c not in identity_cols and c not in teammate_cols]


def linear_cols(cols):
    # the rule we used for logistic regression before v5: no diff_ columns (they are p1 - p2),
    # no active_boost_sum (sum of the single boosts) and no healed (hp_total + damage_taken - healed = 6)
    bad = ["p1_active_boost_sum", "p2_active_boost_sum", "p1_healed", "p2_healed"]
    return [c for c in cols if not c.startswith("diff_") and c not in bad]


X3_lin = X[linear_cols(our_cols)]                  # v3: our features
X4_lin = X[linear_cols(our_cols + identity_cols)]  # v4: + Pokemon identity
X5_lin = select_features(X, "linear")             # v5: + Enes's features, some groups removed
X5_tree = select_features(X, "tree")              # all 217 columns for the tree models
print("v3", X3_lin.shape[1], " v4", X4_lin.shape[1], " v5", X5_lin.shape[1], " tree", X5_tree.shape[1])

v3 61  v4 121  v5 143  tree 217


## Helpers

For every fold we save the predicted probabilities. Then we can compute accuracy, try ensembles
and try thresholds without training the models again. All models see exactly the same 15 folds,
so we can also count in how many folds one model is better than another.

In [4]:
folds = list(make_cv().split(X, y))


def fold_probs(model, X_):
    # probability that P1 wins, for the test part of every fold
    probs = []
    for tr, te in folds:
        model.fit(X_.iloc[tr], y.iloc[tr])
        probs.append(model.predict_proba(X_.iloc[te])[:, 1])
    return probs


def fold_acc(probs, threshold=0.5):
    return np.array([((p > threshold) == y.iloc[te]).mean() for p, (tr, te) in zip(probs, folds)])


def show(name, acc, base=None):
    line = f"{name:32s} {acc.mean():.4f} +- {acc.std():.4f}"
    if base is not None:
        line += f"   diff {acc.mean() - base.mean():+.4f}, better in {(acc > base).sum()}/15 folds"
    print(line)


lr = lambda C=1.0: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
hgb = lambda: HistGradientBoostingClassifier(learning_rate=0.03313073862781126, max_iter=575, max_leaf_nodes=19,
                                             min_samples_leaf=290, l2_regularization=0.12563152773938666,
                                             random_state=42)

## 1. Feature versions (logistic regression)

Same model, more information. For v3 and v4 we used the default C like we did at that time.
In v5 there are many more columns, so we use a smaller C (stronger regularization).

In [5]:
P = {}
P["v3"] = fold_probs(lr(), X3_lin)
P["v4"] = fold_probs(lr(), X4_lin)
P["v5"] = fold_probs(lr(0.02), X5_lin)
A = {k: fold_acc(v) for k, v in P.items()}
show("v3 LR (our features)", A["v3"])
show("v4 LR (+ identity)", A["v4"], A["v3"])
show("v5 LR (+ Enes's features)", A["v5"], A["v4"])

v3 LR (our features)             0.6711 +- 0.0046
v4 LR (+ identity)               0.6757 +- 0.0079   diff +0.0046, better in 12/15 folds
v5 LR (+ Enes's features)        0.6843 +- 0.0068   diff +0.0087, better in 12/15 folds


**Result:** Pokemon identity added +0.005 and Enes's features another +0.009, both better in 12 of 15 folds.
The biggest gain came from new information, not from a new model.

## 2. Did removing feature groups help?

After merging the two feature sets we removed some groups for logistic regression
(the lists in `select_features`). Here we compare with keeping all of them.
Note: we chose these groups with this same CV, so the gain is a bit optimistic.

In [6]:
X5_all_lin = X[linear_cols(list(X.columns))]
P["v5 all groups"] = fold_probs(lr(0.02), X5_all_lin)
A["v5 all groups"] = fold_acc(P["v5 all groups"])
show(f"LR, all groups ({X5_all_lin.shape[1]} cols)", A["v5 all groups"])
show(f"LR, groups removed ({X5_lin.shape[1]} cols)", A["v5"], A["v5 all groups"])

LR, all groups (196 cols)        0.6798 +- 0.0064
LR, groups removed (143 cols)    0.6843 +- 0.0068   diff +0.0045, better in 13/15 folds


**Result:** Removing the groups helped by +0.005 (13/15 folds). More features is not always better:
some of our columns repeated information that was already in Enes's columns.

## 3. Other models on the v5 features

Trees get all 217 columns, the linear models get the 143 linear columns.
Random forest uses the settings we found with random search before.

In [7]:
P["HistGB"] = fold_probs(hgb(), X5_tree)
P["RandomForest"] = fold_probs(RandomForestClassifier(n_estimators=300, min_samples_leaf=11, max_features=0.48,
                                                      random_state=42, n_jobs=-1), X5_tree)
P["NaiveBayes"] = fold_probs(make_pipeline(StandardScaler(), GaussianNB()), X5_lin)
P["LDA"] = fold_probs(make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
                      X5_lin)
for k in ["HistGB", "RandomForest", "NaiveBayes", "LDA"]:
    A[k] = fold_acc(P[k])
show("LogReg", A["v5"])
for k in ["HistGB", "RandomForest", "NaiveBayes", "LDA"]:
    show(k, A[k], A["v5"])

LogReg                           0.6843 +- 0.0068
HistGB                           0.6773 +- 0.0060   diff -0.0071, better in 4/15 folds
RandomForest                     0.6689 +- 0.0060   diff -0.0154, better in 1/15 folds
NaiveBayes                       0.6243 +- 0.0096   diff -0.0600, better in 0/15 folds
LDA                              0.6844 +- 0.0050   diff +0.0001, better in 8/15 folds


**Result:** Logistic regression is the best single model. LDA gives the same score, because it also
draws a straight line between wins and losses. Random forest is clearly worse. Naive Bayes is the worst (0.624):
it treats features as independent, but many of them measure almost the same thing (HP left, fainted, damage),
so it counts the same evidence several times.

## 4. Ensembles

We average the probabilities of two models. The idea: logistic regression and trees make
different mistakes, so the average should be better than both.

In [8]:
def mix(weights):
    total = sum(weights.values())
    return [sum(w * P[k][i] for k, w in weights.items()) / total for i in range(len(folds))]


options = {
    "LR 1 : HistGB 1": {"v5": 1, "HistGB": 1},
    "LR 2 : HistGB 1": {"v5": 2, "HistGB": 1},
    "LR 1 : HistGB 2": {"v5": 1, "HistGB": 2},
    "LR 2 : HistGB 1 : RF 1": {"v5": 2, "HistGB": 1, "RandomForest": 1},
    "LR 2 : HistGB 1 : LDA 1": {"v5": 2, "HistGB": 1, "LDA": 1},
}
show("LogReg alone", A["v5"])
for name, w in options.items():
    show(name, fold_acc(mix(w)), A["v5"])

LogReg alone                     0.6843 +- 0.0068
LR 1 : HistGB 1                  0.6862 +- 0.0069   diff +0.0019, better in 10/15 folds
LR 2 : HistGB 1                  0.6875 +- 0.0048   diff +0.0032, better in 12/15 folds
LR 1 : HistGB 2                  0.6845 +- 0.0070   diff +0.0002, better in 8/15 folds
LR 2 : HistGB 1 : RF 1           0.6871 +- 0.0048   diff +0.0028, better in 10/15 folds
LR 2 : HistGB 1 : LDA 1          0.6879 +- 0.0048   diff +0.0036, better in 11/15 folds


**Result:** LR 2 : HistGB 1 is the best simple ensemble (+0.003, better in 12/15 folds).
Adding random forest or LDA does not really change it (LDA +0.0004 is noise), so we kept two models.

## 5. Threshold

We predict "P1 wins" when the probability is above 0.5. The classes are almost balanced,
so 0.5 should be right, but we check it.

In [9]:
final = mix({"v5": 2, "HistGB": 1})
for t in [0.44, 0.46, 0.48, 0.50, 0.52, 0.54, 0.56]:
    print(f"threshold {t:.2f}: {fold_acc(final, t).mean():.4f}")

threshold 0.44: 0.6855
threshold 0.46: 0.6863
threshold 0.48: 0.6878
threshold 0.50: 0.6875
threshold 0.52: 0.6861
threshold 0.54: 0.6836
threshold 0.56: 0.6801


**Result:** 0.48 and 0.50 are almost the same (difference 0.0003, inside the noise), so we kept 0.5.

## 6. Overfitting check

We train on the whole training set and look at the accuracy on the same data.
A big gap between train and CV means the model memorizes.

In [10]:
m_lr = lr(0.02).fit(X5_lin, y)
m_hgb = hgb().fit(X5_tree, y)
p_lr, p_hgb = m_lr.predict_proba(X5_lin)[:, 1], m_hgb.predict_proba(X5_tree)[:, 1]
rows = [("LogReg", (p_lr > 0.5) == y, A["v5"]),
        ("HistGB", (p_hgb > 0.5) == y, A["HistGB"]),
        ("Ensemble 2:1", ((2 * p_lr + p_hgb) / 3 > 0.5) == y, fold_acc(final))]
for name, train_hits, cv in rows:
    print(f"{name:14s} train {train_hits.mean():.4f}   CV {cv.mean():.4f}   gap {train_hits.mean() - cv.mean():.4f}")

LogReg         train 0.6985   CV 0.6843   gap 0.0142
HistGB         train 0.8581   CV 0.6773   gap 0.1808
Ensemble 2:1   train 0.7620   CV 0.6875   gap 0.0745


**Result:** Logistic regression does not overfit (gap 0.014). HistGB memorizes a lot (gap 0.18) even with
large leaves, but in the ensemble this matters less: the ensemble is still the best in CV.